# 7.3 Inference Sessions — Apply Notebook

## Objective

Master the ONNX Runtime `InferenceSession` API: creation, configuration,
input/output handling, dynamic batching, error handling, and profiling.

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Build a toy ONNX model and create a session | `onnx.helper`, `InferenceSession` |
| 2 | SessionOptions: optimization, threading, arena | `SessionOptions` knobs |
| 3 | IO binding and input/output inspection | `get_inputs`, `get_outputs` |
| 4 | Dynamic batch inference sweep | Variable-size inputs |
| 5 | Error handling for invalid inputs | Graceful failure patterns |
| 6 | Multiple output handling | Named output selection |
| 7 | Session profiling | `enable_profiling`, trace analysis |
| 8 | **Challenge:** build a batched inference server | Production-ready class |

```
pip install onnx onnxruntime numpy
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, json, tempfile, warnings
warnings.filterwarnings("ignore")

import numpy as np

import onnx
from onnx import checker, helper, TensorProto, numpy_helper

import onnxruntime as ort

print(f"ONNX         : {onnx.__version__}")
print(f"ORT          : {ort.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Providers    : {ort.get_available_providers()}")

WORK_DIR = tempfile.mkdtemp(prefix="ort_sessions_")
print(f"Working dir  : {WORK_DIR}")

## Exercise 1 — Build a Toy ONNX Model and Create an InferenceSession

We build a minimal model from scratch using `onnx.helper`:

$$Y = \text{ReLU}(X W + B)$$

where $X \in \mathbb{R}^{N \times 8}$, $W \in \mathbb{R}^{8 \times 4}$,
$B \in \mathbb{R}^{4}$, and $N$ is a dynamic batch dimension.

Then we create an `InferenceSession` and run inference.

In [ ]:
def build_toy_model(path, in_features=8, out_features=4, opset=17):
    """Build Y = ReLU(X @ W + B) as an ONNX model."""
    rng = np.random.default_rng(0)
    W = rng.standard_normal((in_features, out_features)).astype(np.float32)
    B = rng.standard_normal((out_features,)).astype(np.float32)

    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", in_features])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["N", out_features])

    graph = helper.make_graph(
        nodes=[
            helper.make_node("MatMul", ["X", "W"], ["XW"]),
            helper.make_node("Add", ["XW", "B"], ["S"]),
            helper.make_node("Relu", ["S"], ["Y"]),
        ],
        name="ToyLinearRelu",
        inputs=[X],
        outputs=[Y],
        initializer=[
            numpy_helper.from_array(W, name="W"),
            numpy_helper.from_array(B, name="B"),
        ],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
    checker.check_model(model)
    onnx.save(model, path)
    return model


toy_path = os.path.join(WORK_DIR, "toy_linear_relu.onnx")
toy_proto = build_toy_model(toy_path)
print(f"Model saved to: {toy_path}")
print(f"Nodes: {[n.op_type for n in toy_proto.graph.node]}")

# Create session and run
sess = ort.InferenceSession(toy_path, providers=["CPUExecutionProvider"])
x = np.random.randn(5, 8).astype(np.float32)
result = sess.run(None, {"X": x})

print(f"\nInput shape  : {x.shape}")
print(f"Output shape : {result[0].shape}")
print(f"Output sample: {result[0][0]}")

# Manual verification: ReLU(X @ W + B)
W = numpy_helper.to_array(toy_proto.graph.initializer[0])
B = numpy_helper.to_array(toy_proto.graph.initializer[1])
expected = np.maximum(0, x @ W + B)
assert np.allclose(result[0], expected, atol=1e-6)
print(f"Manual parity: max|Δ| = {np.abs(result[0] - expected).max():.2e} ✓")

## Exercise 2 — SessionOptions: Optimization, Threading, Memory Arena

`SessionOptions` controls how ORT prepares and runs the model.

| Option | Effect |
|--------|--------|
| `graph_optimization_level` | DISABLED / BASIC / EXTENDED / ALL |
| `intra_op_num_threads` | Parallelism within a single operator |
| `inter_op_num_threads` | Parallelism across independent operators |
| `enable_mem_pattern` | Pre-allocate memory based on graph analysis |
| `enable_cpu_mem_arena` | Use an arena allocator for CPU tensors |
| `execution_mode` | SEQUENTIAL or PARALLEL |

In [ ]:
configs = {
    "Default": {},
    "Disabled Opt": {
        "graph_optimization_level": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    },
    "All Opt + 1 thread": {
        "graph_optimization_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
        "intra_op_num_threads": 1,
        "inter_op_num_threads": 1,
    },
    "All Opt + 4 threads": {
        "graph_optimization_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
        "intra_op_num_threads": 4,
        "inter_op_num_threads": 2,
    },
    "No Arena": {
        "enable_cpu_mem_arena": False,
        "enable_mem_pattern": False,
    },
}

x_bench = np.random.randn(64, 8).astype(np.float32)

print(f"{'Config':<24} {'Median (ms)':>12} {'p95 (ms)':>12}")
print("─" * 50)

for name, opts in configs.items():
    so = ort.SessionOptions()
    so.log_severity_level = 3
    for k, v in opts.items():
        setattr(so, k, v)

    s = ort.InferenceSession(toy_path, so, providers=["CPUExecutionProvider"])

    # Warmup
    for _ in range(50):
        s.run(None, {"X": x_bench})

    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        s.run(None, {"X": x_bench})
        times.append(time.perf_counter() - t0)

    arr = np.array(times) * 1000
    print(f"{name:<24} {np.median(arr):>10.4f} {np.percentile(arr, 95):>10.4f}")

print("\nNote: for this tiny model, differences may be negligible.")

## Exercise 3 — IO Binding and Input/Output Inspection

Before running inference, inspect what the session expects and produces.
`get_inputs()` and `get_outputs()` return `NodeArg` objects with name, shape,
and type metadata.

In [ ]:
sess = ort.InferenceSession(toy_path, providers=["CPUExecutionProvider"])

print("Inputs:")
for inp in sess.get_inputs():
    print(f"  name={inp.name!r:12} shape={str(inp.shape):<16} type={inp.type}")

print("\nOutputs:")
for out in sess.get_outputs():
    print(f"  name={out.name!r:12} shape={str(out.shape):<16} type={out.type}")

# Model metadata
meta = sess.get_modelmeta()
print(f"\nModel metadata:")
print(f"  producer     : {meta.producer_name}")
print(f"  graph_name   : {meta.graph_name}")
print(f"  description  : {meta.description or '(empty)'}")
print(f"  domain       : {meta.domain or '(empty)'}")
print(f"  version      : {meta.version}")

# Providers used
print(f"\nActive providers: {sess.get_providers()}")

# Run with named output selection
x = np.random.randn(3, 8).astype(np.float32)
y_all   = sess.run(None, {"X": x})            # all outputs
y_named = sess.run(["Y"], {"X": x})            # specific output by name

assert np.array_equal(y_all[0], y_named[0])
print(f"\nNamed output selection works: shapes match ✓")

## Exercise 4 — Dynamic Batch Inference Sweep

Since our model has `N` as a symbolic dimension, we can run any batch size.
We sweep from 1 to 1024 and measure per-sample latency.

$$t_{\text{per-sample}}(N) = \frac{t_{\text{total}}(N)}{N}$$

Typically per-sample cost **decreases** with larger batches due to
amortised overhead (session call, memory allocation).

In [ ]:
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.log_severity_level = 3
sess = ort.InferenceSession(toy_path, so, providers=["CPUExecutionProvider"])

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
results = []

print(f"{'Batch':>6} {'Total (ms)':>12} {'Per-sample (µs)':>16} {'Output shape':>14}")
print("─" * 52)

for bs in batch_sizes:
    x = np.random.randn(bs, 8).astype(np.float32)

    # Warmup
    for _ in range(30):
        sess.run(None, {"X": x})

    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        out = sess.run(None, {"X": x})
        times.append(time.perf_counter() - t0)

    arr = np.array(times)
    total_ms = np.median(arr) * 1000
    per_sample_us = total_ms * 1000 / bs
    results.append((bs, total_ms, per_sample_us))

    assert out[0].shape == (bs, 4), f"Unexpected shape at batch={bs}"
    print(f"{bs:>6} {total_ms:>10.4f} {per_sample_us:>14.2f} {str(out[0].shape):>14}")

# Verify amortisation
first_per = results[0][2]
last_per  = results[-1][2]
print(f"\nPer-sample cost:  bs=1 → {first_per:.2f} µs,  bs=1024 → {last_per:.2f} µs")
print(f"Amortisation factor: {first_per / last_per:.1f}x")

## Exercise 5 — Error Handling for Invalid Inputs

Robust inference code must handle:
- Wrong dtype
- Wrong shape (wrong number of features)
- Wrong input name
- Missing inputs

ORT raises `ort.capi.onnxruntime_pybind11_state.InvalidArgument` (or similar)
for these cases.

In [ ]:
sess = ort.InferenceSession(toy_path, providers=["CPUExecutionProvider"])

error_cases = [
    ("Wrong dtype (float64)",    {"X": np.random.randn(2, 8).astype(np.float64)}),
    ("Wrong features (5 vs 8)",  {"X": np.random.randn(2, 5).astype(np.float32)}),
    ("Wrong input name",         {"Z": np.random.randn(2, 8).astype(np.float32)}),
    ("3-D instead of 2-D",      {"X": np.random.randn(2, 4, 2).astype(np.float32)}),
]

print(f"{'Error Case':<28} {'Result':<40}")
print("─" * 70)

for label, feed in error_cases:
    try:
        sess.run(None, feed)
        print(f"{label:<28} Unexpected success")
    except Exception as e:
        err_type = type(e).__name__
        msg = str(e)[:80]
        print(f"{label:<28} {err_type}: {msg}...")

# Correct input
ok = sess.run(None, {"X": np.random.randn(2, 8).astype(np.float32)})
print(f"\nCorrect input → output shape: {ok[0].shape} ✓")

## Exercise 6 — Multiple Output Handling

Build a model with **two outputs**: the ReLU result and the pre-activation
logits.  Show how to request specific outputs by name.

In [ ]:
def build_multi_output_model(path, in_f=8, out_f=4, opset=17):
    rng = np.random.default_rng(42)
    W = rng.standard_normal((in_f, out_f)).astype(np.float32)
    B = rng.standard_normal((out_f,)).astype(np.float32)

    X_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["N", in_f])
    logits_info = helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["N", out_f])
    activated_info = helper.make_tensor_value_info("activated", TensorProto.FLOAT, ["N", out_f])

    graph = helper.make_graph(
        nodes=[
            helper.make_node("MatMul", ["X", "W"], ["XW"]),
            helper.make_node("Add", ["XW", "B"], ["logits"]),
            helper.make_node("Relu", ["logits"], ["activated"]),
        ],
        name="MultiOutput",
        inputs=[X_info],
        outputs=[logits_info, activated_info],
        initializer=[
            numpy_helper.from_array(W, name="W"),
            numpy_helper.from_array(B, name="B"),
        ],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])
    checker.check_model(model)
    onnx.save(model, path)
    return model


multi_path = os.path.join(WORK_DIR, "multi_output.onnx")
build_multi_output_model(multi_path)

sess = ort.InferenceSession(multi_path, providers=["CPUExecutionProvider"])

print("Outputs available:")
for o in sess.get_outputs():
    print(f"  {o.name}: shape={o.shape}")

x = np.random.randn(4, 8).astype(np.float32)

# All outputs
all_out = sess.run(None, {"X": x})
print(f"\nAll outputs: {len(all_out)} tensors")
print(f"  logits shape   : {all_out[0].shape}")
print(f"  activated shape: {all_out[1].shape}")

# Selective: only logits
logits_only = sess.run(["logits"], {"X": x})
assert np.array_equal(logits_only[0], all_out[0])
print(f"\nSelective (logits only): shape={logits_only[0].shape} ✓")

# Selective: only activated
act_only = sess.run(["activated"], {"X": x})
assert np.array_equal(act_only[0], all_out[1])
print(f"Selective (activated only): shape={act_only[0].shape} ✓")

# Verify relationship: activated = ReLU(logits)
expected_act = np.maximum(0, all_out[0])
assert np.allclose(all_out[1], expected_act, atol=1e-7)
print(f"\nVerified: activated == ReLU(logits) ✓")

## Exercise 7 — Session Profiling

ORT can dump a per-node execution profile (Chrome-trace JSON format).
This lets you identify which operators consume the most time.

Enable with `so.enable_profiling = True`, then call
`sess.end_profiling()` to get the trace file path.

In [ ]:
so = ort.SessionOptions()
so.enable_profiling = True
so.profile_file_prefix = os.path.join(WORK_DIR, "ort_profile")
so.log_severity_level = 3

sess = ort.InferenceSession(toy_path, so, providers=["CPUExecutionProvider"])

x = np.random.randn(128, 8).astype(np.float32)
for _ in range(50):
    sess.run(None, {"X": x})

profile_path = sess.end_profiling()
print(f"Profile saved to: {profile_path}")

# Parse the profile
with open(profile_path) as f:
    events = json.load(f)

# Extract kernel execution events
kernel_events = [
    e for e in events
    if isinstance(e, dict) and e.get("cat") == "Node" and "dur" in e
]

if kernel_events:
    print(f"\nKernel events found: {len(kernel_events)}")
    print(f"{'Op':<20} {'Count':>6} {'Total (µs)':>12} {'Avg (µs)':>12}")
    print("─" * 54)

    from collections import defaultdict
    op_times = defaultdict(list)
    for e in kernel_events:
        op_name = e.get("name", "unknown")
        op_times[op_name].append(e["dur"])

    for op, durations in sorted(op_times.items(), key=lambda x: -sum(x[1])):
        total = sum(durations)
        avg = total / len(durations)
        print(f"{op:<20} {len(durations):>6} {total:>10.1f} {avg:>10.2f}")
else:
    print("\nProfile events:")
    for e in events[:10]:
        if isinstance(e, dict):
            print(f"  cat={e.get('cat', '?'):<16} name={e.get('name', '?'):<20} dur={e.get('dur', '?')}")

## Exercise 8 — Challenge: Build a Batched Inference Server

Create a production-ready `BatchedInferenceServer` class that:

1. Loads an ONNX model with configurable `SessionOptions`.
2. Validates inputs against the model schema.
3. Supports dynamic batching with automatic input/output name resolution.
4. Collects latency statistics (median, p95, p99).
5. Has a warm-up method.

In [ ]:
class BatchedInferenceServer:
    """Production-ready wrapper around ORT InferenceSession."""

    DTYPE_MAP = {
        "tensor(float)": np.float32,
        "tensor(double)": np.float64,
        "tensor(int64)": np.int64,
        "tensor(int32)": np.int32,
    }

    def __init__(self, model_path, intra_threads=None, inter_threads=1,
                 opt_level="ALL"):
        so = ort.SessionOptions()
        so.log_severity_level = 3

        levels = {
            "DISABLED": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
            "BASIC":    ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
            "EXTENDED": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
            "ALL":      ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
        }
        so.graph_optimization_level = levels.get(opt_level, levels["ALL"])

        if intra_threads:
            so.intra_op_num_threads = intra_threads
        so.inter_op_num_threads = inter_threads

        self.sess = ort.InferenceSession(
            model_path, so, providers=["CPUExecutionProvider"]
        )
        self._input_specs = {
            inp.name: {"shape": inp.shape, "type": inp.type}
            for inp in self.sess.get_inputs()
        }
        self._output_names = [o.name for o in self.sess.get_outputs()]
        self._latencies = []

    @property
    def input_names(self):
        return list(self._input_specs.keys())

    @property
    def output_names(self):
        return self._output_names

    def validate_input(self, feed):
        """Check that feed dict matches model schema."""
        errors = []
        for name, spec in self._input_specs.items():
            if name not in feed:
                errors.append(f"Missing input: {name!r}")
                continue
            arr = feed[name]
            expected_dtype = self.DTYPE_MAP.get(spec["type"])
            if expected_dtype and arr.dtype != expected_dtype:
                errors.append(f"{name}: dtype {arr.dtype} != expected {expected_dtype}")
            for i, (got, exp) in enumerate(zip(arr.shape, spec["shape"])):
                if isinstance(exp, int) and got != exp:
                    errors.append(f"{name}: dim[{i}] = {got} != {exp}")
        return errors

    def predict(self, feed, output_names=None):
        """Run inference with validation and latency tracking."""
        errors = self.validate_input(feed)
        if errors:
            raise ValueError(f"Input validation failed: {'; '.join(errors)}")

        t0 = time.perf_counter()
        result = self.sess.run(output_names, feed)
        self._latencies.append(time.perf_counter() - t0)
        return result

    def warmup(self, n=50):
        """Run warmup iterations with random data."""
        feed = {}
        for name, spec in self._input_specs.items():
            shape = [s if isinstance(s, int) else 1 for s in spec["shape"]]
            dtype = self.DTYPE_MAP.get(spec["type"], np.float32)
            feed[name] = np.random.randn(*shape).astype(dtype)
        for _ in range(n):
            self.sess.run(None, feed)

    def stats(self):
        if not self._latencies:
            return {}
        arr = np.array(self._latencies) * 1000
        return {
            "count": len(arr),
            "median_ms": float(np.median(arr)),
            "p95_ms": float(np.percentile(arr, 95)),
            "p99_ms": float(np.percentile(arr, 99)),
            "mean_ms": float(arr.mean()),
        }

    def reset_stats(self):
        self._latencies.clear()


# ── Test the server ────────────────────────────────────────────────────
server = BatchedInferenceServer(toy_path, intra_threads=2)
print(f"Inputs : {server.input_names}")
print(f"Outputs: {server.output_names}")

server.warmup(100)
print(f"Warmup complete.")

# Valid predictions
for bs in [1, 8, 32, 128, 512]:
    x = np.random.randn(bs, 8).astype(np.float32)
    out = server.predict({"X": x})
    assert out[0].shape == (bs, 4)

# Invalid input
try:
    server.predict({"X": np.random.randn(2, 5).astype(np.float32)})
except ValueError as e:
    print(f"\nValidation caught: {e}")

# Statistics
stats = server.stats()
print(f"\nLatency stats ({stats['count']} calls):")
print(f"  median: {stats['median_ms']:.4f} ms")
print(f"  p95   : {stats['p95_ms']:.4f} ms")
print(f"  p99   : {stats['p99_ms']:.4f} ms")

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| Model building | `onnx.helper` constructs models from scratch — no training framework needed |
| SessionOptions | Control optimization level, threading, memory arena, and execution mode |
| IO inspection | `get_inputs` / `get_outputs` return name, shape, and type metadata |
| Dynamic batch | Symbolic dims accept any size at runtime; per-sample cost drops with larger batches |
| Error handling | ORT raises specific exceptions for dtype, shape, and name mismatches |
| Multiple outputs | Request specific outputs by name to skip unnecessary computation |
| Profiling | `enable_profiling` produces Chrome-trace JSON for per-node timing |
| Server class | Wrap session with validation, warmup, and latency tracking for production |